## ___Updating the mycorrhizal states___
--------------------

In [1]:
!python --version

Python 3.13.8


The system cannot find the path specified.


In [2]:
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

In [12]:
# https://datadryad.org/dataset/doi:10.5061/dryad.n8bm9
# THE SHEET "Original states data" HAS THE RAW DATA SCRAPED FROM PUBLICATIONS WITHOUT ANY INTEFERENCE FROM THE AUTHORS!!!!
maherali_original = pd.read_excel(r"../../data/chapter2/Maherali.etal.AmNat.Data.xlsx", sheet_name="Original states data", skiprows=range(2),
                                  usecols=("Source", "Original name (Genus species)", "Raw state record from publication"))
maherali_original.rename(mapper={old: old.replace('(', '').replace(')', '').lower().replace(' ', '_') for old in maherali_original.columns}, axis=1, inplace=True) # column names have parentheses and spaces
# taxonomy columns in Maherali et. al. dataset has trailing spaces :(
maherali_original.loc[:, "original_name_genus_species"] = maherali_original.original_name_genus_species.str.strip()
maherali_original.loc[:, "raw_state_record_from_publication"] = maherali_original.raw_state_record_from_publication.str.strip()
maherali_original.drop_duplicates(subset=("original_name_genus_species", "raw_state_record_from_publication"), inplace=True)

# final_maherali = pd.read_excel(r"../../data/chapter2/Maherali.etal.AmNat.Data.xlsx", sheet_name="Final list matched with phylo", skiprows=range(2))
# final_maherali.rename(mapper={old: old.lower().replace(' ', '_') for old in final_maherali.columns}, axis=1, inplace=True)
# final_maherali.genus_species = final_maherali.genus_species.str.strip().str.replace('_', ' ') # the sheet "Final list matched with phylo" has genus and specific epithets concatenated by under scores!

# in TRY, mycorrhiza type is trait id 7
try_myco = pd.read_csv(r"../../data/chapter2/TRY/mycorrhizal_states.txt", delimiter='\t', low_memory=False, encoding="latin1", usecols=["Dataset", "SpeciesName", "AccSpeciesName", "OrigValueStr",
                                "TraitID"]).dropna(subset=["AccSpeciesName", "OrigValueStr", "TraitID"])
# unify the mycorrhizal state info
# 'ECTO', 'NM/AM', 'EC', 'EC/AM', 'AM', 'Ecto', 'Non',        'vesicular-arbuscular mycorrhiza', 'ectomycorrhiza', 'no', '0', 'Ph.th.end.', 'VAM', 'Ectomycorrhiza', 'E.ch.ect.', 'arbuscular',
# 'ec?', 'VA', 'ecto', 'Absent', 'non-ectomycorrhizal', 'ectomycorrhizal', 'Yes', 'No', 'EM', 'AMNM', 'NM', 'AM + EM', 'ERM', 'Ericoid', 'ECM'

MYCORRHIZAL_STATES_REPLACEMENTS = {
    "ECTO": "EcM",
    "Ecto": "EcM",
    "EC": "EcM",
    "ectomycorrhiza": "EcM",
    "Ectomycorrhiza": "EcM",
    "ecto": "EM",
    "ectomycorrhizal": "EcM",
    "ECM": "EcM",
    "vesicular-arbuscular mycorrhiza" : "AM",
    "VAM": "AM",
    "VA": "AM",
    "Non": "NM",
    "AMNM": "NM/AM",
    "Ericoid": "ER",
    "ERM": "ER",
    "AM + EM": "AM/EcM",
    "EC/AM": "AM/EcM",
    "Orchid": "OrM",
    "OrM": "OrM"
}

try_myco.loc[:, "OrigValueStr"] = try_myco.OrigValueStr.replace(MYCORRHIZAL_STATES_REPLACEMENTS)
try_myco = try_myco.query("OrigValueStr.isin(@MYCORRHIZAL_STATES_REPLACEMENTS.values())")

mycodb_v4 = pd.read_csv(r"../../data/chapter2/MycoDB_version4.csv", usecols=["PlantSpecies2018", "FUNGROUP", "MYCORRHIZAETYPE", "AM_single_genus", "EM_single_genus", 
                        "STERILIZED", "NONMYCOCONTROL", "NONMYCOCONTROL2"]).dropna(subset="PlantSpecies2018").drop_duplicates()
mycodb_v4.loc[:, "PlantSpecies2018"] = mycodb_v4.PlantSpecies2018.str.capitalize().str.replace('_', ' ')

# scrape the online only MycoDB metadata and serialize it to the disk
# req = Request(url=r"https://www.nature.com/articles/sdata201628/tables/2", headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:142.0) Gecko/20100101 Firefox/142.0"})
# with urlopen(req) as r:
#     soup = BeautifulSoup(r.read())
# 
# table = soup.find(name="table", attrs={"class": "data last-table"}) # locate the metadata table
# [th.text.strip() for th in table.find_all(name="th")] # column names
# mycodb_descriptions = [[td.text for td in tr.find_all(name="td")] for tr in table.find_all(name="tr")[1:]] # parse the rows
# 
# # create a dataframe using the parsed rows and column names and serialize it to the disk
# pd.DataFrame({ 
#     "Variable": [row[0] for row in mycodb_descriptions],
#     "Description": [row[1] for row in mycodb_descriptions],
#     "Variable Type (range)": [row[2] for row in mycodb_descriptions],
#     "Levels (#studies/level)": [row[3] for row in mycodb_descriptions],
# }).to_csv(r"../data/chapter2/MycoDB_version4_metadata.csv", index=False)

mycodb_v4_meta = pd.read_csv(r"../../data/chapter2/MycoDB_version4_metadata.csv")

subset_categorical = pd.read_csv(r"../../data/chapter2/FREDv3subset/FRED_subset_categorical.csv")

# this only has genus level mycorrhizal types
fungalroot = pd.read_csv(r"../../data/chapter2/FungalRoot/FungalRoot_cleaned.csv", low_memory=False, encoding="utf-8")
fungalroot.loc[:, "species"] = fungalroot.species.str.strip()

In [15]:
# we are not just looking to fill the missing mycorrhizal state info here!
# we could potentially find and reconcile conflicts between information in FRED and other databases!
pd.merge(left=subset_categorical, left_on="binominal", right=maherali_original, right_on="original_name_genus_species", how="left")

,binominal,F01286,F01287,F01289,F01290,F00043,F00645,F00004,source,original_name_genus_species,raw_state_record_from_publication
0,Populus trichocarpa,Populus,trichocarpa,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Wang&Qiu2006,Populus trichocarpa,EM
1,Populus trichocarpa,Populus,trichocarpa,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Hempel et al. 2013,Populus trichocarpa,AM+EM
2,Populus tremula,Populus,tremula,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Wang&Qiu2006,Populus tremula,AM + EM
3,Populus tremula,Populus,tremula,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...",Akhmetzhanova et al. 2012,Populus tremula,EM
4,Altingia obovata,Altingia,obovata,Altingiaceae,Saxifragales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi...",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
261,Populus deltoides,Populus,deltoides,Salicaceae,Malpighiales,C3,NaN,Valverde et al (unpublished),Akhmetzhanova et al. 2012,Populus deltoides,EM
262,Prunus sargentii,Prunus,sargentii,Rosaceae,Rosales,C3,NaN,Valverde et al (unpublished),NaN,NaN,NaN
263,Styphnolobium japonicum,Styphnolobium,japonicum,Fabaceae,Fabales,C3,NaN,Valverde et al (unpublished),NaN,NaN,NaN
264,Syringa reticulata,Syringa,reticulata,Oleaceae,Lamiales,C3,NaN,Valverde et al (unpublished),NaN,NaN,NaN


In [16]:
# same for FungalRoot too. 
# instead of just populating the empty cells, look for conflicts between data that exists in FRED and FungalRoot!
pd.merge(left=subset_categorical, left_on="binominal", right=fungalroot, right_on="species")

,binominal,F01286,F01287,F01289,F01290,F00043,F00645,F00004,original_reference,original_ref_checked,...,mycorrhiza type,remark_mycorrhiza_ type,curator_remark_1_name,curator_remark_1_comment,curator_remark_2_name,curator _remark_2_comment,curator_remark_3_name,curator_remark_3_comment,other_remarks,consensus_mycorrhizal_state
0,Populus trichocarpa,Populus,trichocarpa,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...","Akhmetzhanova, A. A., Soudzilovskaia, N. A., O...",NaN,...,EcM; others not addressed,NaN,NaN,NaN,NaN,NaN,Laura M. Suz,ok,NaN,EcM-AM
1,Populus trichocarpa,Populus,trichocarpa,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...","Schultz, R. C., Isebrands, J. G., & Kormanik, ...",NaN,...,EcM; no others,NaN,NaN,NaN,NaN,NaN,Laura M. Suz,ok,NaN,EcM-AM
2,Populus trichocarpa,Populus,trichocarpa,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...","Baum, C., & Makeschin, F. (2000). Effects of n...",y,...,EcM; others not addressed,NaN,NaN,NaN,NaN,NaN,Laura M. Suz,ok,NaN,EcM-AM
3,Populus trichocarpa,Populus,trichocarpa,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...","Trappe, J. (1962). Bot. Rev., 23, 528-606.",y,...,EcM; others not addressed,NaN,Leho Tedersoo,in this study 26% reports conflict with predic...,NaN,NaN,Laura M. Suz,ok,NaN,EcM-AM
4,Populus trichocarpa,Populus,trichocarpa,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord...","Harley, J. L., & Brierley, J. K. (1954). The u...",y,...,EcM; others not addressed,NaN,NaN,NaN,NaN,NaN,Laura M. Suz,ok,NaN,EcM-AM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
897,Ulmus americana,Ulmus,americana,Ulmaceae,Rosales,C3,NaN,Valverde et al (unpublished),"Brundrett, M., Murase, G., & Kendrick, B. (199...",NaN,...,AM; no others,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AM
898,Ulmus americana,Ulmus,americana,Ulmaceae,Rosales,C3,NaN,Valverde et al (unpublished),"McDougall, W. B. (1914). On the mycorrhizas of...",NaN,...,non-mycorrhizal (checked for all types),NaN,Leho Tedersoo,probably incorrect (this genus is AM; in this ...,NaN,NaN,NaN,NaN,NaN,AM
899,Ulmus americana,Ulmus,americana,Ulmaceae,Rosales,C3,NaN,Valverde et al (unpublished),"Thomas Jr, W. D. (1943). Mycorrhizae associate...",NaN,...,EcM; no others,NaN,Leho Tedersoo,incorrect report (this genus is AM; in this st...,NaN,NaN,Laura M. Suz,not ECM,NaN,AM
900,Ulmus americana,Ulmus,americana,Ulmaceae,Rosales,C3,NaN,Valverde et al (unpublished),"Vozzo, J. A., & Hacskaylo, E. (1964). Anatomy ...",NaN,...,AM; no others,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AM


In [20]:
 subset_categorical.F00645.unique()

array([nan, 'AM', 'EM', 'AM + EM', 'NM', 'ErM'], dtype=object)

In [23]:
try_myco.OrigValueStr.unique()

array(['NM/AM', 'EM', 'AM/EM', 'ER', 'AM', 'NM', 'OM'], dtype=object)

In [22]:
pd.merge(left=subset_categorical, left_on="binominal", right=try_myco, right_on="AccSpeciesName", how="inner").drop_duplicates(subset=["binominal", "OrigValueStr"]).OrigValueStr.unique()

array(['EM', 'NM', 'AM', 'NM/AM', 'AM/EM', 'ER'], dtype=object)